# 04 — Entrenamiento de baselines formales

Compara dos entrenamientos con la misma arquitectura y protocolo. La única diferencia es augmentation en train; validation permanece limpia y test no participa en entrenamiento, early stopping ni checkpoint.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FORMAL_DIR = PROJECT_ROOT / 'outputs/experiments/baseline_formal'
VARIANTS = {
    'Con aumento': FORMAL_DIR / 'baseline_con_aumento',
    'Sin aumento': FORMAL_DIR / 'baseline_sin_aumento',
}
VARIANTS

Comandos reproducibles ejecutados desde la raíz del proyecto:

```bash
python -m src.training.train --experiment-name baseline_formal/baseline_con_aumento --augmentation --epochs 10 --batch-size 64 --learning-rate 0.0001 --patience 5 --seed 42
python -m src.training.train --experiment-name baseline_formal/baseline_sin_aumento --no-augmentation --epochs 10 --batch-size 64 --learning-rate 0.0001 --patience 5 --seed 42
```

In [ ]:
configs = {}
histories = {}
for name, directory in VARIANTS.items():
    configs[name] = json.loads((directory / 'training_config.json').read_text(encoding='utf-8'))
    histories[name] = pd.read_csv(directory / 'history.csv')
display(pd.DataFrame(configs).T[['train_augmentation', 'epochs_completed', 'batch_size', 'learning_rate', 'seed']])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, history in histories.items():
    epochs = history['epoch'] + 1
    axes[0].plot(epochs, history['loss'], label=f'{name} — train')
    axes[0].plot(epochs, history['val_loss'], '--', label=f'{name} — validation')
    axes[1].plot(epochs, history['val_binary_accuracy'], label=name)
axes[0].set(title='Loss por época', xlabel='Época', ylabel='BCE')
axes[1].set(title='Accuracy de validation', xlabel='Época', ylabel='Accuracy')
for ax in axes: ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout()

Los `.keras` se generan localmente y quedan fuera de Git. Ejecutar las celdas en orden; el notebook no guarda outputs pesados.